# 8/2 Improve delta calculation for FlatVol model

---

In BS model the spot at time t is $ S_t = S_0 e^{(r-\frac{\sigma^2}{2})t+\sigma W_t} $. Knowing this, when we bump the spot we dont have to resimulate the spot, instead we can obtain it by scaling the inital simulation: $ S^{bumped}_t = S_0*(1+\delta) e^{(r-\frac{\sigma^2}{2})t+\sigma W_t} $.

a, implement this delta calculation method for MonteCarlo pricer (10p)

b, compare the value and the calculation time of the improved delta with the default bump and revaluation delta. (5p)


In [3]:
import time
import numpy as np
import sys
from pathlib import Path

_cwd = Path.cwd()
_assignment_root = _cwd if (_cwd / "src").exists() else _cwd / "homeworks" / "Assignment 7"
if (_assignment_root / "src").exists() and str(_assignment_root) not in sys.path:
    sys.path.insert(0, str(_assignment_root))

from src.market_data import MarketData
from src.enums import Stock, PutCallFwd, LongShort, MCNumMethod
from src.contract import EuropeanContract
from src.model import FlatVolModel
from src.numerical_method import MCMethodFlatVol, MCParams

In [4]:
MarketData.initialize()

In [5]:
und = Stock.TEST_COMPANY
spot = MarketData.get_spot()[und]
rate = MarketData.get_risk_free_rate()
strike = spot  # ATM
expiry = 1.0

contract = EuropeanContract(und, PutCallFwd.CALL, LongShort.LONG, strike, expiry)
model = FlatVolModel(und)
vol = model.get_vol(strike, expiry)

In [6]:
und = Stock.TEST_COMPANY
spot = MarketData.get_spot()[und]
rate = MarketData.get_risk_free_rate()
strike = spot
expiry = 1.0

contract = EuropeanContract(und, PutCallFwd.CALL, LongShort.LONG, strike, expiry)
model = FlatVolModel(und)

In [7]:
params = MCParams(
    seed=476,
    num_of_path=50_000,
    tenor_frequency=52,
    standardize=False,
    antithetic=False,
    control_variate=False,
    evolve_spot_method=MCNumMethod.EXACT,
)

BUMP = 0.01  # 1% spot bump

In [9]:
def calc_pv_call(
    spot_paths,
    strike,
    rate,
    expiry
):
    S_T = spot_paths[:, -1] #shape: (num_paths, num_tenors)
    payoffs = np.maximum(S_T - strike, 0.0)
    return np.exp(-rate * expiry) * np.mean(payoffs)

In [10]:
def calc_delta_pathscaling(
    contract: EuropeanContract,
    model: FlatVolModel,
    params: MCParams,
    bump,
):
    """Simulate once, then scale all paths by (1 + bump).

    Because S_t = S_0 * exp(...) is linear in S_0:
        S_t^bumped = S_0 * (1 + bump) * exp(...) = S_t * (1 + bump)
    """
    mc = MCMethodFlatVol(contract, model, params)
    spot_paths = mc.simulate_spot_paths()

    pv_base = calc_pv_call(spot_paths, contract.strike, rate, contract.expiry)
    pv_bumped = calc_pv_call(spot_paths * (1 + bump), contract.strike, rate, contract.expiry)

    delta = (pv_bumped - pv_base) / (model.spot * bump)
    return delta, pv_base

delta, pv = calc_delta_pathscaling(contract, model, params, BUMP)
print(f"Delta (path scaling): {delta:.4f}, PV: {pv:.4f}")

Delta (path scaling): 0.6313, PV: 14.1984


In [12]:
def calc_delta_bump_reprice(
    contract: EuropeanContract,
    model: FlatVolModel,
    params: MCParams,
    bump,
):
    """Bump spot in the model, re-simulate from scratch, reprice."""
    mc_base = MCMethodFlatVol(contract, model, params)
    spot_paths_base = mc_base.simulate_spot_paths()
    pv_base = calc_pv_call(spot_paths_base, contract.strike, rate, contract.expiry)

    bump_size = model.spot * bump
    model.bump_spot(bump_size)
    mc_bumped = MCMethodFlatVol(contract, model, params)

    spot_paths_bumped = mc_bumped.simulate_spot_paths()

    pv_bumped = calc_pv_call(spot_paths_bumped, contract.strike, rate, contract.expiry)
    model.bump_spot(-bump_size)  # restore model

    delta = (pv_bumped - pv_base) / bump_size
    return delta, pv_base

In [14]:
count = 3

times_ps, times_br = [], []

for _ in range(count):
    start = time.perf_counter()
    delta_ps, pv_ps = calc_delta_pathscaling(contract, model, params, BUMP)
    times_ps.append(time.perf_counter() - start)

for _ in range(count):
    start = time.perf_counter()
    delta_br, pv_br = calc_delta_bump_reprice(contract, model, params, BUMP)
    times_br.append(time.perf_counter() - start)

avg_ps = np.mean(times_ps)
avg_br = np.mean(times_br)

print(f"{'Method':<22} {'Delta':>10} {'PV':>10} {'Avg time':>10}")
print(f"{'Path scaling':<22} {delta_ps:>10.6f} {pv_ps:>10.4f} {avg_ps:>10.4f}s")
print(f"{'Bump & reprice':<22} {delta_br:>10.6f} {pv_br:>10.4f} {avg_br:>10.4f}s")
print(f"{'Speedup':<22} {'':>21} {avg_br / avg_ps:>9.1f}x")

Method                      Delta         PV   Avg time
Path scaling             0.631336    14.1984   135.6711s
Bump & reprice           0.631336    14.1984   267.3761s
Speedup                                            2.0x
